<a href="https://colab.research.google.com/github/ManideepLadi/cs6910_assignment3/blob/manideep/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dakshina Dataset from google


In [24]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [25]:
!wget https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar

--2021-04-29 15:11:52--  https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.5.128, 74.125.206.128, 64.233.167.128, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.5.128|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2008340480 (1.9G) [application/x-tar]
Saving to: ‘dakshina_dataset_v1.0.tar.1’

dakshina_dataset_v1 100%[===================>]   1.87G  34.0MB/s    in 19s     

2021-04-29 15:12:11 (103 MB/s) - ‘dakshina_dataset_v1.0.tar.1’ saved [2008340480/2008340480]



In [26]:
!tar -xvf '/content/dakshina_dataset_v1.0.tar'

dakshina_dataset_v1.0/bn/
dakshina_dataset_v1.0/bn/lexicons/
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.test.tsv
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.train.tsv
dakshina_dataset_v1.0/bn/lexicons/bn.translit.sampled.dev.tsv
dakshina_dataset_v1.0/bn/native_script_wikipedia/
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.valid.text.shuf.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.info.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.info.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.text.sorted.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-filt.train.text.shuf.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.nonblock.sections.tsv.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.omit_pages.txt.gz
dakshina_dataset_v1.0/bn/native_script_wikipedia/bn.wiki-full.text.sorted.tsv.gz
dakshina_dataset_v1.0/bn/na

Preprocess data

In [27]:
# import string
 
# # load doc into memory
# def load_doc(filename):
# 	# open the file as read only
# 	file = open(filename, mode='rt', encoding='utf-8')
# 	# read all text
# 	text = file.read()
# 	# close the file
# 	file.close()
# 	return text
 
# # split a loaded document into sentences
# def to_pairs(doc):
# 	lines = doc.strip().split('\n')
# 	pairs = [line.split('\t') for line in  lines]
# 	return pairs


In [28]:

# load dataset
filename = 'dakshina_dataset_v1.0/te/lexicons/te.translit.sampled.train.tsv'
# doc = load_doc(filename)
# # split into english-german pairs
# pairs = to_pairs(doc)

In [29]:
# # Vectorize the data.
# input_characters = set()
# target_characters = set()
# for pair in pairs:
#   for char in pair[1]:
#     if pair[1] not in input_characters:
#       input_characters.add(char)
#   for char in pair[0]:
#     if char not in target_characters:
#       target_characters.add(char)

# input_characters = sorted(list(input_characters))
# target_characters = sorted(list(target_characters))
# num_encoder_tokens = len(input_characters)
# num_decoder_tokens = len(target_characters)


# print("Number of unique input tokens:", num_encoder_tokens)
# print("Number of unique output tokens:", num_decoder_tokens)


In [30]:
# Vectorize the data.
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(filename, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: len(lines) - 1]:
    target_text,input_text, attestation = line.split("\t")
    # We use "tab" as the "start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    for i in range(int(attestation)):
      input_texts.append(input_text)
      target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

Number of samples: 84680
Number of unique input tokens: 26
Number of unique output tokens: 65
Max sequence length for inputs: 25
Max sequence length for outputs: 22


In [31]:
input_texts[1]

'ankita'

In [32]:
target_texts[1]

'\tఅంకిత\n'

In [33]:
input_characters[3]

'd'

In [34]:
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32"
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32"
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

In [35]:
target_texts[0]

'\tఅంకిత\n'

In [36]:
encoder_input_data[0,6]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [37]:
decoder_input_data[0,6]

array([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
      dtype=float32)

In [38]:
num_encoder_tokens

26

In [41]:
class RNN_Model:

  def __init__(self,input_embedding_size,no_of_encoder_layers,no_of_decoder_layers,latent_dimension,dropout,beam_size):
    self.input_embedding_size = input_embedding_size
    self.no_of_encoder_layers = no_of_encoder_layers
    self.no_of_decoder_layers = no_of_decoder_layers
    self.latent_dimension = latent_dimension
    self.dropout = dropout
    self.beam_size = beam_size
    self.model = None

  def printModelParameters(self):
    print(self.input_embedding_size)
    print(self.no_of_encoder_layers)
    print(self.no_of_decoder_layers)
    print(self.latent_dimension)
    print(self.dropout)

  def BUILD_MODEL(self,num_encoder_tokens,num_decoder_tokens):
    
    #Build multilayer encoder
    encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
    input_to_encoder=encoder_inputs
    encoder_states=None
    encoder_lstmLayers=[]
    for i in range(self.no_of_encoder_layers):
      encoder_lstm_layer.append(keras.layers.LSTM(self.latent_dimension, return_state=True, return_sequences=True)
      e_outputs, h, c = encoder_lstm_layer(input_to_encoder) 
      input_to_encoder = e_outputs
      if i==self.no_of_encoder_layers-1:
        encoder_states = [h,c]


    # Set up the multi decoder, using `encoder_states` as initial state.
    decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))
    d_outputs, inititalh,initialc = keras.layers.LSTM(self.latent_dimension, return_state=True, return_sequences=True)(decoder_inputs,initial_state=encoder_states)
    input_to_decoder = decoder_inputs
    final_decoder_outputs=None
    for i in range(self.no_of_decoder_layers-1):
      decoder_lstm_layer = keras.layers.LSTM(self.latent_dimension, return_state=True, return_sequences=True)
      decoder_outputs, h, c = decoder_lstm_layer(input_to_decoder)
      input_to_decoder = decoder_outputs
      if i==self.no_of_decoder_layers-2:
        final_decoder_outputs=decoder_outputs
         
    decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
    dense_decoder_outputs = decoder_dense(final_decoder_outputs)

    # Define the model that will turn
    # `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
    self.model = keras.Model([encoder_inputs, decoder_inputs], dense_decoder_outputs)
    self.model.compile(optimizer='rmsprop',loss='categorical_crossentropy',metrics=['accuracy'])
    return


  #Fit model using the train datagenerator and returns fitted model..
  def fit_RNN(self , encoder_input_data,decoder_input_data ,decoder_target_data,epochs ,batch_size):
    self.model.fit(
        [encoder_input_data, decoder_input_data],
        decoder_target_data,
        batch_size=64,
        epochs=30,
        validation_split=0.1)
        # validation_split=0.1,
        # callbacks = [WandbCallback(monitor='val_accuracy',
        #                                             save_model = True))
    return

In [42]:
rnn = RNN_Model(32,3,3,64,0.3,0)
rnn.BUILD_MODEL(num_encoder_tokens,num_decoder_tokens)
rnn.printModelParameters()
rnn.model.summary()
rnn.fit_RNN( encoder_input_data,decoder_input_data,  decoder_target_data,
    batch_size=64,
    epochs=10)

32
3
3
64
0.3
Model: "model_5"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_13 (InputLayer)           [(None, None, 65)]   0                                            
__________________________________________________________________________________________________
lstm_35 (LSTM)                  [(None, None, 64), ( 33280       input_13[0][0]                   
__________________________________________________________________________________________________
lstm_36 (LSTM)                  [(None, None, 64), ( 33024       lstm_35[0][0]                    
__________________________________________________________________________________________________
input_12 (InputLayer)           [(None, None, 26)]   0                                            
______________________________________________________________________________

In [45]:
latent_dims = [1024, 512,  256]
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))

outputs = encoder_inputs
encoder_states = []
for j in range(len(latent_dims))[::-1]:
    outputs, h, c = keras.layers.LSTM(latent_dims[j], return_state=True, return_sequences=bool(j))(outputs)
    encoder_states += [h, c]

# Set up the decoder, setting the initial state of each layer to the state of the layer in the encoder
# which is it's mirror (so for encoder: a->b->c, you'd have decoder initial states: c->b->a).
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

outputs = decoder_inputs
output_layers = []
for j in range(len(latent_dims)):
    output_layers.append(
        keras.layers.LSTM(latent_dims[len(latent_dims) - j - 1], return_sequences=True, return_state=True)
    )
    outputs, dh, dc = output_layers[-1](outputs, initial_state=encoder_states[2*j:2*(j+1)])


decoder_dense = keras.layers.Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

Model: "model_6"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_15 (InputLayer)           [(None, None, 26)]   0                                            
__________________________________________________________________________________________________
input_16 (InputLayer)           [(None, None, 65)]   0                                            
__________________________________________________________________________________________________
lstm_37 (LSTM)                  [(None, None, 256),  289792      input_15[0][0]                   
__________________________________________________________________________________________________
lstm_40 (LSTM)                  [(None, None, 256),  329728      input_16[0][0]                   
                                                                 lstm_37[0][1]              

In [38]:
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
print(encoder_inputs.shape)
# encoder1 = keras.layers.LSTM(64, return_state=True)

# encoder_outputs1, state_h1, state_c1 = encoder1(encoder_inputs)

# print(encoder_outputs1.shape)


# encoder2=keras.layers.LSTM(64, return_state=True)
# encoder_outputs=keras.layers.Reshape((None,None,64), input_shape=encoder_outputs1.shape)
# print(encoder_outputs.shape)
# encoder_outputs2, state_h2, state_c2 = encoder2(encoder_outputs)

# # We discard `encoder_outputs` and only keep the states.
# encoder_states = [state_h1, state_c1]encoder_inputs = Input(shape=(None, num_encoder_tokens))

e_outputs, h1, c1 = keras.layers.LSTM(64, return_state=True, return_sequences=True)(encoder_inputs) 
print(e_outputs.shape)
_, h2, c2 = keras.layers.LSTM(64, return_state=True)(e_outputs) 
encoder_states = [h1, c1, h2, c2]


# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = keras.layers.LSTM(64, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=[h2,c2])
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.summary()

(None, None, 26)
(None, None, 64)
Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_9 (InputLayer)            [(None, None, 26)]   0                                            
__________________________________________________________________________________________________
lstm_13 (LSTM)                  [(None, None, 64), ( 23296       input_9[0][0]                    
__________________________________________________________________________________________________
input_10 (InputLayer)           [(None, None, 65)]   0                                            
__________________________________________________________________________________________________
lstm_14 (LSTM)                  [(None, 64), (None,  33024       lstm_13[0][0]                    
__________________________________________________________

In [39]:
encoder_input_data.shape

(84680, 25, 26)

In [40]:
decoder_input_data.shape

(84680, 22, 65)

In [41]:
decoder_target_data.shape


(84680, 22, 65)

In [54]:
model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=64,
    epochs=10,
    validation_split=0.1
)
# Save model
model.save("s2s")

Epoch 1/10
1191/1191 [==============================] - 186s 148ms/step - loss: 0.0136 - accuracy: 0.3922 - val_loss: 0.5598 - val_accuracy: 0.2946
Epoch 2/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0112 - accuracy: 0.3929 - val_loss: 0.5441 - val_accuracy: 0.2981
Epoch 3/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0100 - accuracy: 0.3936 - val_loss: 0.5575 - val_accuracy: 0.2981
Epoch 4/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0090 - accuracy: 0.3938 - val_loss: 0.5670 - val_accuracy: 0.2970
Epoch 5/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0081 - accuracy: 0.3926 - val_loss: 0.6189 - val_accuracy: 0.2958
Epoch 6/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0075 - accuracy: 0.3937 - val_loss: 0.5842 - val_accuracy: 0.2970
Epoch 7/10
1191/1191 [==============================] - 175s 147ms/step - loss: 0.0068 - accuracy: 0.3936 - val_

INFO:tensorflow:Assets written to: s2s/assets


INFO:tensorflow:Assets written to: s2s/assets


In [ ]:
model = keras.models.load_model("s2s")

encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_outputs2, state_h_enc2, state_c_enc2 = model.layers[4].output  # lstm_2
encoder_states = [state_h_enc2, state_c_enc2]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(256,), name="input_3")
decoder_state_input_c = keras.Input(shape=(256,), name="input_4")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())


def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [ ]:
for seq_index in range(200):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index +100: seq_index + 101]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: amkita
Decoded sentence: అంచనాలతో

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలు

-
Input sentence: ankitam
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitabaavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankatamicchaadu
Decoded sentence: అంచున

-
Input sentence: ankitamicchadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamicchaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamichhaadu
Decoded sentence: అంచుని

-
Input sentence: amkithamichaaru
Decoded senten

In [48]:
# Define sampling models (modified for n-layer deep network)
encoder_model = keras.Model(encoder_inputs, encoder_states)


d_outputs = decoder_inputs
decoder_states_inputs = []
decoder_states = []
for j in range(len(latent_dims))[::-1]:
    current_state_inputs = [keras.Input(shape=(latent_dims[j],)) for _ in range(2)]

    temp = output_layers[len(latent_dims)-j-1](d_outputs, initial_state=current_state_inputs)

    d_outputs, cur_states = temp[0], temp[1:]

    decoder_states += cur_states
    decoder_states_inputs += current_state_inputs

decoder_outputs = decoder_dense(d_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

In [52]:
reverse_input_char_index = dict(
    (i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict(
    (i, char) for char, i in target_token_index.items())
def decode_sequence(input_seq, encoder_model, decoder_model):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index['\t']] = 1.

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = []  #Creating a list then using "".join() is usually much faster for string creation
    while not stop_condition:
        to_split = decoder_model.predict([target_seq] + states_value)

        output_tokens, states_value = to_split[0], to_split[1:]

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, 0])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence.append(sampled_char)

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == '\n' or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

    return "".join(decoded_sentence)

In [53]:
for seq_index in range(200):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index +100: seq_index + 101]
    decoded_sentence = decode_sequence(input_seq,encoder_model,decoder_model)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: amkita
Decoded sentence: అంచనాలతో

-
Input sentence: ankita
Decoded sentence: అంచనలను

-
Input sentence: ankita
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలను

-
Input sentence: ankitha
Decoded sentence: అంచనాలు

-
Input sentence: ankitam
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitham
Decoded sentence: అంచనాలు

-
Input sentence: ankitabaavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankithabhavam
Decoded sentence: అంచు

-
Input sentence: ankatamicchaadu
Decoded sentence: అంచున

-
Input sentence: ankitamicchadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచున

-
Input sentence: ankitamichhaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamicchaadu
Decoded sentence: అంచుని

-
Input sentence: ankithamichhaadu
Decoded sentence: అంచుని

-
Input sentence: amkithamichaaru
Decoded sentenc

In [ ]:
!pip install wandb -qqq
import wandb
wandb.login()

In [ ]:
sweep_config = {
  'name': 'RNN',
  'method': 'grid',
  'metric': {
      'name': 'accuracy',
      'goal': 'maximize'   
    },
  'parameters': {
        'input_embedding_size': {
            'values': [16, 32, 64, 256]
        },
        'encoder_layers':{
            'values':[1,2,3]
        },
        'decoder_layers':{
            'values':[1,2,3]
        },
        'hidden_layer_size':{
            'values':[16, 32, 64, 256]
        },
        'cell_type':{
            'values':['RNN', 'GRU', 'LSTM']
        },
        'dropout':{
            'values':[0.3,0.2]
        },
        'beam_sizes':{
            'values':['No','Yes']
        }

    }
}

sweep_id = wandb.sweep(sweep_config, project='RNN', entity='manideepladi')